# RAV-Med — Step 4: verifier and relevance baselines

**Question this notebook answers.** Is the finding of the paper specific to the general-domain verifier (`roberta-large-mnli`) and the SBERT similarity, or does it hold when the contradiction term and the relevance term are replaced by other, including biomedical, models? It also tests two NLI-only baselines that use no similarity at all (entailment margin E − C).

**Same data as the paper.** PubMedQA: the 4,000 passages of `rav_med_full_results_N1000.csv`. HealthContradict: the top-ranked 80-word window of each document, chosen by the SBERT similarity exactly as in Step 2 (the notebook checks that it reproduces your stored similarity values).

**Safeguards built in**
1. *Reproduction check first.* The notebook re-scores `roberta-large-mnli` and compares with your stored contradiction scores. If the premise/hypothesis order or truncation differs, it says so and stops. Nothing new is trusted until the old numbers are reproduced.
2. *Label-order check for every NLI model.* Label names are read from each model's config, and every model is also tested on 9 hand-written medical sentence pairs (3 contradictions, 3 entailments, 3 neutral). A model whose labels cannot be mapped consistently is skipped, never guessed.
3. *Resumable.* Every model's scores are cached to disk (Google Drive if you allow it), so a disconnect loses at most one model.
4. *Model versions recorded* (Hugging Face commit hashes) for the paper.

**Upload when asked (files you already have):** `rav_med_full_results_N1000.csv` (main PubMedQA results), `hc_claims.csv` and `hc_doc_scores.csv` (from the Step 2 zip).
**Runtime:** GPU → T4. About 30–60 minutes in total on a free T4 (more if a model has to be re-downloaded). If a model fails to download, the notebook continues with the others and tells you which one failed.

In [ ]:
!pip install -q sentence-transformers transformers accelerate scikit-learn sentencepiece protobuf huggingface_hub

In [ ]:
import os, json, hashlib, subprocess, random, time, numpy as np, pandas as pd, torch
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"; print("device:",DEVICE)
assert DEVICE=="cuda", "Switch the runtime to a GPU (Runtime > Change runtime type > T4 GPU)"
EMBED_MODEL="pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"      # as in the paper (max_seq_length 100)
CHUNK_WORDS, MIN_CHUNK_WORDS, MAX_CHUNKS = 80, 15, 120                     # identical to Step 2
NLI_MAXLEN, REL_MAXLEN, BS = 256, 512, 32
# candidate NLI verifiers (first = the paper's own verifier, re-scored in the same pipeline). Models that fail to load are skipped.
NLI_MODELS=["roberta-large-mnli","microsoft/deberta-large-mnli","MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli","pritamdeka/PubMedBERT-MNLI-MedNLI","cross-encoder/nli-deberta-v3-large"]
REL_MODELS=["ncbi/MedCPT-Cross-Encoder","cross-encoder/ms-marco-MiniLM-L-6-v2"]
# cache (Drive survives disconnects)
CACHE="/content/step4_cache"
try:
    from google.colab import drive; drive.mount("/content/drive"); CACHE="/content/drive/MyDrive/ravmed_step4_cache"
except Exception as e: print("Drive not mounted, using local cache:", e)
os.makedirs(CACHE, exist_ok=True); print("cache:", CACHE)
for f in ["rav_med_full_results_N1000.csv","hc_claims.csv","hc_doc_scores.csv"]:
    if not os.path.exists(f):
        from google.colab import files; print("Upload", f); files.upload()
    assert os.path.exists(f), "missing "+f

## 1. Build the items (PubMedQA passages and HealthContradict windows)

In [ ]:
from sentence_transformers import SentenceTransformer
pq=pd.read_csv("rav_med_full_results_N1000.csv"); assert len(pq)==4000 and set(pq.condition)=={"CLEAN","DRIFT","NOISE","CONTRADICTION"}
pq=pq.rename(columns={"similarity":"S","contradiction":"C_stored"}).reset_index(drop=True)
if not os.path.exists("HealthContradict"): subprocess.run(["git","clone","--depth","1","https://github.com/tinaboya/HealthContradict"],check=True)
raw=pd.DataFrame([json.loads(l) for l in open("HealthContradict/dataset/dataset_ready.jsonl",encoding="utf-8")]); raw["topic_id"]=raw.topic_id.astype(int)
did=lambda t: hashlib.md5(t.encode("utf-8")).hexdigest()[:12]
docs={}; 
for _,r in raw.iterrows():
    for t in (r.doc_a,r.doc_b): docs[did(t)]=t
cor=np.where(raw.query_stance=="yes",raw.doc_a,raw.doc_b); inc=np.where(raw.query_stance=="yes",raw.doc_b,raw.doc_a)
pairs=pd.DataFrame(dict(topic_id=raw.topic_id,ans=raw.query_stance,cid=[did(t) for t in cor],iid=[did(t) for t in inc])); print("pairs:",len(pairs),"topics:",pairs.topic_id.nunique())
claims=pd.read_csv("hc_claims.csv").set_index("topic_id"); sc=pd.read_csv("hc_doc_scores.csv").drop_duplicates(["topic_id","doc_id","claim_type"]).set_index(["topic_id","doc_id","claim_type"])
need=sorted(set(zip(pairs.topic_id,pairs.cid))|set(zip(pairs.topic_id,pairs.iid)))
emb=SentenceTransformer(EMBED_MODEL,device=DEVICE); print("SBERT max_seq_length:",emb.max_seq_length); assert emb.max_seq_length==100
def chunk(text):
    w=text.split(); ch=[" ".join(w[i:i+CHUNK_WORDS]) for i in range(0,len(w),CHUNK_WORDS)]
    if len(ch)>1 and len(ch[-1].split())<MIN_CHUNK_WORDS: ch=ch[:-1]
    return ch[:MAX_CHUNKS] if ch else [text]
cemb={(t,ct):emb.encode([claims.loc[t,f"claim_{ct}"]],normalize_embeddings=True,convert_to_numpy=True)[0] for t in claims.index for ct in ("ref","free")}
doc_chunks={}; doc_emb={}
for d in sorted({d for _,d in need}):
    doc_chunks[d]=chunk(docs[d]); doc_emb[d]=emb.encode(doc_chunks[d],normalize_embeddings=True,convert_to_numpy=True,batch_size=64)
rows=[]
for t,d in need:
    for ct in ("ref","free"):
        S=doc_emb[d]@cemb[(t,ct)]; k=int(np.argmax(S))
        rows.append(dict(topic_id=t,doc_id=d,claim_type=ct,window_index=k,text=doc_chunks[d][k],claim=claims.loc[t,f"claim_{ct}"],S=float(S[k]),S_step2=float(sc.loc[(t,d,ct),"S_top1"]),C_stored=float(sc.loc[(t,d,ct),"C_top1"])))
hc=pd.DataFrame(rows); diff=(hc.S-hc.S_step2).abs(); print("HC items:",len(hc),"| max |S - S_step2| =",round(float(diff.max()),6))
assert diff.max()<=1e-3, "HealthContradict windows do not reproduce Step 2 - stop and send this output"
pq.to_pickle(f"{CACHE}/pq_items.pkl"); hc.to_pickle(f"{CACHE}/hc_items.pkl")
del emb; torch.cuda.empty_cache()

## 2. NLI verifiers (with reproduction and label-order checks)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import HfApi
PROBES=[("contradiction","Insulin lowers blood glucose in patients with diabetes.","Insulin does not lower blood glucose in patients with diabetes."),
("contradiction","The trial found that aspirin reduced the risk of heart attack.","Aspirin increased the risk of heart attack."),
("contradiction","Antibiotics are ineffective against viral infections.","Antibiotics cure viral infections."),
("entailment","Metformin is a first-line drug for type 2 diabetes and lowers blood glucose.","Metformin lowers blood glucose."),
("entailment","The patient was admitted with chest pain and diagnosed with myocardial infarction.","The patient had a heart attack."),
("entailment","Smoking cessation reduced mortality in the cohort.","Quitting smoking was associated with lower death rates."),
("neutral","The study enrolled 200 adults with asthma.","Inhaled steroids reduce exacerbations."),
("neutral","The hospital opened a new cardiology wing in 2019.","Statins lower cholesterol."),
("neutral","Vitamin D levels were measured in winter.","The participants were mostly women.")]
@torch.no_grad()
def nli_probs(tok,model,prem,hyp,bs=BS):
    order=np.argsort([len(p) for p in prem]); out=np.zeros((len(prem),model.config.num_labels),dtype=np.float32)
    for i in range(0,len(prem),bs):
        idx=order[i:i+bs]; enc=tok([prem[j] for j in idx],[hyp[j] for j in idx],truncation=True,padding=True,max_length=NLI_MAXLEN,return_tensors="pt").to(DEVICE)
        with torch.autocast("cuda",dtype=torch.float16): lg=model(**enc).logits
        out[idx]=torch.softmax(lg.float(),-1).cpu().numpy()
    return out
def label_map(model,tok):
    """class -> index. If the config names all three classes, use them, but the contradiction index must be confirmed by the probes.
    Otherwise (generic LABEL_0..2) the mapping must come from consistent, distinct probe votes; if not, return None (model skipped)."""
    names={int(i):str(l).lower() for i,l in model.config.id2label.items()}; cfg={}
    for i,l in names.items():
        for c,key in [("contradiction","contra"),("entailment","entail"),("neutral","neutral")]:
            if l.startswith(key): cfg[c]=i
    P=nli_probs(tok,model,[p[1] for p in PROBES],[p[2] for p in PROBES])
    votes={c:P[[k for k,p in enumerate(PROBES) if p[0]==c]].argmax(1) for c in ("contradiction","entailment","neutral")}
    if len(cfg)==3:
        if (votes["contradiction"]==cfg["contradiction"]).sum()<2: return None,names,f"config says contradiction={cfg['contradiction']} but probes disagree {votes['contradiction'].tolist()}"
        return cfg,names,"ok (config names, contradiction confirmed by probes)"
    pm={c:int(np.bincount(v,minlength=P.shape[1]).argmax()) for c,v in votes.items()}
    if (votes["contradiction"]==pm["contradiction"]).sum()<2: return None,names,"contradiction probes not consistent: "+str(votes["contradiction"].tolist())
    if pm["contradiction"] in (pm["entailment"],pm["neutral"]) and len({pm["entailment"],pm["neutral"]})==2: return None,names,"contradiction index collides with another class: "+str(pm)
    ok_en=(votes["entailment"]==pm["entailment"]).sum()>=2 and (votes["neutral"]==pm["neutral"]).sum()>=2 and len(set(pm.values()))==3
    if not ok_en:
        print("   WARNING: entailment/neutral indices could not be resolved from the probes; the entailment-margin baseline is omitted for this model (contradiction index is confirmed).")
        return {"contradiction":pm["contradiction"],"entailment":None,"neutral":None},names,"ok (contradiction inferred from probes; E/N unresolved)"
    return pm,names,"ok (inferred from probes)"
def repro_check(res,tok=None,model=None):
    dpq=res["pq"].C.values-pq.C_stored.values; dhc=res["hc"].C.values-hc.C_stored.values
    rp=np.corrcoef(res["pq"].C,pq.C_stored)[0,1]; rh=np.corrcoef(res["hc"].C,hc.C_stored)[0,1]
    print(f"REPRODUCTION  PubMedQA: mean|diff|={np.abs(dpq).mean():.4f} max={np.abs(dpq).max():.3f} r={rp:.5f} | HealthContradict: mean|diff|={np.abs(dhc).mean():.4f} max={np.abs(dhc).max():.3f} r={rh:.5f}")
    ok=(rp>0.999) and np.abs(dpq).mean()<0.01 and (rh>0.999)
    if not ok and tok is not None and model is not None:
        pm_,_,_=label_map(model,tok); sw=nli_probs(tok,model,pq.claim.tolist()[:300],pq.passage.tolist()[:300])[:,pm_["contradiction"]]
        print("  swapped premise/hypothesis on 300 PubMedQA rows: r =",round(float(np.corrcoef(sw,pq.C_stored.values[:300])[0,1]),5),"(if this is ~1, the main run used the opposite order)")
    print("REPRODUCTION:","PASSED" if ok else "FAILED - stop here and send me this output"); assert ok, "stored scores not reproduced"
info={}; nli_scores={}
for name in NLI_MODELS:
    f=f"{CACHE}/nli_{name.replace('/','__')}.pkl"
    if os.path.exists(f):
        nli_scores[name]=pd.read_pickle(f); info[name]=nli_scores[name].pop("info",dict(status="ok (cached, info missing)")); print("cached:",name)
        if name=="roberta-large-mnli": repro_check(nli_scores[name])
        continue
    t0=time.time()
    try:
        tok=AutoTokenizer.from_pretrained(name); model=AutoModelForSequenceClassification.from_pretrained(name).to(DEVICE).eval()
        try: sha=HfApi().model_info(name).sha
        except Exception: sha="unknown"
        pm,names,why=label_map(model,tok)
        if pm is None: print("SKIP",name,"-",why,names); info[name]=dict(status="skipped: "+why,labels=names); del model; torch.cuda.empty_cache(); continue
        print(name,"label map:",pm,"| config labels:",names)
        rp=nli_probs(tok,model,pq.passage.tolist(),pq.claim.tolist()); rh=nli_probs(tok,model,hc.text.tolist(),hc.claim.tolist())
        col=lambda a,c: a[:,pm[c]] if pm[c] is not None else np.full(len(a),np.nan)
        res=dict(pq=pd.DataFrame(dict(C=col(rp,"contradiction"),E=col(rp,"entailment"),N=col(rp,"neutral"))),hc=pd.DataFrame(dict(C=col(rh,"contradiction"),E=col(rh,"entailment"),N=col(rh,"neutral"))))
        if name=="roberta-large-mnli": repro_check(res,tok,model)      # nothing is cached unless the paper's own scores are reproduced
        info[name]=dict(status="ok",sha=sha,labels=names,label_map=pm,minutes=round((time.time()-t0)/60,1)); nli_scores[name]=res
        pd.to_pickle({**res,"info":info[name]},f)
        del model; torch.cuda.empty_cache(); print("done",name,info[name]["minutes"],"min")
    except AssertionError: raise
    except Exception as e: print("FAILED",name,"-",repr(e)[:300]); info[name]=dict(status="failed: "+repr(e)[:200])
assert "roberta-large-mnli" in nli_scores, "the paper's verifier must be scored first"


## 3. Relevance baselines (cross-encoders)

In [ ]:
rel_scores={}
@torch.no_grad()
def rel_logits(tok,model,q,t,bs=16):
    order=np.argsort([len(x) for x in t]); out=np.zeros(len(t),dtype=np.float32)
    for i in range(0,len(t),bs):
        idx=order[i:i+bs]; enc=tok([q[j] for j in idx],[t[j] for j in idx],truncation=True,padding=True,max_length=REL_MAXLEN,return_tensors="pt").to(DEVICE)
        with torch.autocast("cuda",dtype=torch.float16): lg=model(**enc).logits
        out[idx]=lg.float()[:,0].cpu().numpy() if lg.shape[-1]==1 else lg.float()[:,-1].cpu().numpy()
    return out
for name in REL_MODELS:
    f=f"{CACHE}/rel_{name.replace('/','__')}.pkl"
    if os.path.exists(f): rel_scores[name]=pd.read_pickle(f); info[name]=rel_scores[name].pop("info",dict(status="ok (cached, info missing)")); print("cached:",name); continue
    try:
        tok=AutoTokenizer.from_pretrained(name); model=AutoModelForSequenceClassification.from_pretrained(name).to(DEVICE).eval()
        try: sha=HfApi().model_info(name).sha
        except Exception: sha="unknown"
        res=dict(pq=rel_logits(tok,model,pq.claim.tolist(),pq.passage.tolist()),hc=rel_logits(tok,model,hc.claim.tolist(),hc.text.tolist()))
        info[name]=dict(status="ok",sha=sha,num_labels=int(model.config.num_labels)); rel_scores[name]=res; pd.to_pickle({**res,"info":info[name]},f); print("done",name,"num_labels",model.config.num_labels)
        del model; torch.cuda.empty_cache()
    except Exception as e: print("FAILED",name,"-",repr(e)[:300]); info[name]=dict(status="failed: "+repr(e)[:200])

## 4. Analysis (same metrics, same bootstrap as the paper: resampling queries for PubMedQA and questions for HealthContradict)

In [ ]:
# ===== Analysis (self-contained; runs on CPU) =====
from sklearn.metrics import roc_auc_score
import numpy as np, pandas as pd, json, hashlib
def sig(x): return 1/(1+np.exp(-np.asarray(x,float)))
def build_methods(df, nli_names, rel_names, has_E=True):
    """returns {method_name: score_array} for a dataframe with columns S, C_<m>, E_<m>, REL_<r>"""
    M={'Similarity only (SBERT)':df['S'].values}
    for r in rel_names: M[f'Relevance only ({r})']=df[f'REL_{r}'].values
    for m in nli_names:
        M[f'Contradiction only, -C ({m})']=-df[f'C_{m}'].values
        M[f'ERS = S - C ({m})']=df['S'].values-df[f'C_{m}'].values
        if not np.isnan(df[f'E_{m}'].values).any(): M[f'Entailment margin = E - C ({m})']=df[f'E_{m}'].values-df[f'C_{m}'].values
    for r in rel_names:
        for m in nli_names[:1]: M[f'sigmoid({r}) - C ({m})']=sig(df[f'REL_{r}'].values)-df[f'C_{m}'].values
    return M
def pq_metrics(cond, y_idx, s):
    """cond: array of condition names; s: score. returns T1,T2,T3 AUROC (positive=CLEAN)"""
    cl=cond=='CLEAN'; out={}
    for nm,cs in [('Conflict',['CONTRADICTION']),('Irrelevance',['DRIFT','NOISE']),('Pooled',['CONTRADICTION','DRIFT','NOISE'])]:
        o=np.isin(cond,cs); y=np.r_[np.ones(cl.sum()),np.zeros(o.sum())]; out[nm]=roc_auc_score(y,np.r_[s[cl],s[o]])
    return out
def pq_table(pq, M, B=500, seed=42):
    rng=np.random.default_rng(seed); pubs=pq.pubid.values; up=np.unique(pubs); rows_of={p:np.where(pubs==p)[0] for p in up}; cond=pq.condition.values
    est={k:pq_metrics(cond,None,v) for k,v in M.items()}
    boots={k:{t:[] for t in ('Conflict','Irrelevance','Pooled')} for k in M}
    for b in range(B):
        ids=np.concatenate([rows_of[p] for p in rng.choice(up,len(up))]); c=cond[ids]
        for k,v in M.items():
            r=pq_metrics(c,None,v[ids])
            for t in r: boots[k][t].append(r[t])
    ref='ERS = S - C (roberta-large-mnli)'
    out=[]
    for k in M:
        row={'Method':k}
        for t in ('Conflict','Irrelevance','Pooled'):
            lo,hi=np.percentile(boots[k][t],[2.5,97.5]); row[t]=f'{est[k][t]:.3f} [{lo:.3f}, {hi:.3f}]'
            if ref in M and k!=ref:
                d=np.array(boots[k][t])-np.array(boots[ref][t]); row[t+' diff vs ref']=f'{est[k][t]-est[ref][t]:+.3f} [{np.percentile(d,2.5):+.3f}, {np.percentile(d,97.5):+.3f}]'
        out.append(row)
    return pd.DataFrame(out)
def hc_pairs(jsonl):
    raw=pd.DataFrame([json.loads(l) for l in open(jsonl,encoding='utf-8')]); raw['topic_id']=raw.topic_id.astype(int)
    did=lambda t:hashlib.md5(t.encode('utf-8')).hexdigest()[:12]
    cor=np.where(raw.query_stance=='yes',raw.doc_a,raw.doc_b); inc=np.where(raw.query_stance=='yes',raw.doc_b,raw.doc_a)
    return pd.DataFrame(dict(topic_id=raw.topic_id,ans=raw.query_stance,cid=[did(t) for t in cor],iid=[did(t) for t in inc]))
def hc_table(hc, pairs, nli_names, rel_names, claim_type, B=1000, seed=42):
    d=hc[hc.claim_type==claim_type].drop_duplicates(['topic_id','doc_id']).set_index(['topic_id','doc_id'])
    dc=d.loc[list(zip(pairs.topic_id,pairs.cid))].reset_index(); di=d.loc[list(zip(pairs.topic_id,pairs.iid))].reset_index()
    Mc=build_methods(dc,nli_names,rel_names); Mi=build_methods(di,nli_names,rel_names)
    topics=pairs.topic_id.values; ut=np.unique(topics); rows_of={t:np.where(topics==t)[0] for t in ut}; rng=np.random.default_rng(seed)
    win={k:(Mc[k]>Mi[k]).astype(float)+0.5*(Mc[k]==Mi[k]) for k in Mc}
    bs={k:[] for k in Mc}
    for b in range(B):
        ids=np.concatenate([rows_of[t] for t in rng.choice(ut,len(ut))])
        for k in Mc: bs[k].append(win[k][ids].mean())
    ref='ERS = S - C (roberta-large-mnli)'; out=[]
    for k in Mc:
        lo,hi=np.percentile(bs[k],[2.5,97.5]); row={'Method':k,f'Pairwise accuracy, {claim_type} claim':f'{100*win[k].mean():.1f} [{100*lo:.1f}, {100*hi:.1f}]'}
        if ref in Mc and k!=ref:
            dd=np.array(bs[k])-np.array(bs[ref]); row['Diff vs ref (points)']=f'{100*(win[k].mean()-win[ref].mean()):+.1f} [{100*np.percentile(dd,2.5):+.1f}, {100*np.percentile(dd,97.5):+.1f}]'
        out.append(row)
    return pd.DataFrame(out)


In [ ]:
short=lambda n:n.split("/")[-1]
nli_names=[short(n) for n in nli_scores]; rel_names=[short(n) for n in rel_scores]
P=pq[["pubid","condition","S"]].copy(); H=hc[["topic_id","doc_id","claim_type","S"]].copy()
for n,r in nli_scores.items():
    for col in "CEN": P[f"{col}_{short(n)}"]=r["pq"][col].values; H[f"{col}_{short(n)}"]=r["hc"][col].values
for n,r in rel_scores.items(): P[f"REL_{short(n)}"]=r["pq"]; H[f"REL_{short(n)}"]=r["hc"]
Mpq=build_methods(P,nli_names,rel_names); tab_pq=pq_table(P,Mpq,B=500)
stored=pq.S.values-pq.C_stored.values
ref_auc=pq_metrics(pq.condition.values,None,stored)["Pooled"]; print("Stored paper ERS, pooled AUROC:",round(ref_auc,3),"(the paper reports 0.822)")
assert abs(ref_auc-0.822)<0.002, "stored data do not match the paper - check you uploaded rav_med_full_results_N1000.csv"
tab_ref=hc_table(H,pairs,nli_names,rel_names,"ref"); tab_free=hc_table(H,pairs,nli_names,rel_names,"free")
pd.set_option("display.width",250,"display.max_colwidth",80)
print("\n=== PubMedQA: AUROC [95% CI over queries] (positive = CLEAN) ==="); print(tab_pq[["Method","Conflict","Irrelevance","Pooled"]].to_string(index=False))
print("\n=== PubMedQA: differences vs ERS with roberta-large-mnli ==="); print(tab_pq[["Method","Conflict diff vs ref","Irrelevance diff vs ref","Pooled diff vs ref"]].to_string(index=False))
print("\n=== HealthContradict, reference claim ==="); print(tab_ref.to_string(index=False))
print("\n=== HealthContradict, label-free claim ==="); print(tab_free.to_string(index=False))

## 5. Save everything

In [ ]:
tab_pq.to_csv("step4_baselines_pubmedqa.csv",index=False); tab_ref.to_csv("step4_baselines_hc_ref.csv",index=False); tab_free.to_csv("step4_baselines_hc_free.csv",index=False)
json.dump(info,open("step4_model_info.json","w"),indent=1); print(json.dumps(info,indent=1)[:3000])
import zipfile
with zipfile.ZipFile("step4_baseline_outputs.zip","w") as z:
    for f in ["step4_baselines_pubmedqa.csv","step4_baselines_hc_ref.csv","step4_baselines_hc_free.csv","step4_model_info.json"]: z.write(f)
    for f in os.listdir(CACHE):
        if f.endswith(".pkl") and f.startswith(("nli_","rel_")): z.write(os.path.join(CACHE,f),"cache/"+f)
try:
    from google.colab import files; files.download("step4_baseline_outputs.zip")
except ImportError: pass